In [1]:
pip install konlpy

   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ---------- ----------------------------- 5.0/19.4 MB 23.6 MB/s eta 0:00:01
   ---------------------- ----------------- 10.7/19.4 MB 26.1 MB/s eta 0:00:01
   ---------------------------------------  19.4/19.4 MB 31.9 MB/s eta 0:00:01
   ---------------------------------------- 19.4/19.4 MB 30.6 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [konlpy]
   ---------------------------------------- 2/2 [konlpy]

Note: you may need to restart the kernel to use updated packages.


In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from openpyxl import Workbook
from konlpy.tag import Okt #OpenKoreantext
from collections import Counter
import time

#셀레니움의 옵션 설정
options = Options()
options.add_argument("--no-sandbox") #사이트 보안정책 방해 받지 않게 하기
options.add_argument("--disable-dev-share-usage") 
#가상의 공간에 저장할 수 있게 하기 / 크롬 웹브라우저에서 주는 가상 메모리 공간에 못 넣는 경우 하드 디스크에 저장해라 
options.add_argument("--headless") #사이트 열지 않게 하기

driver = webdriver.Chrome(options=options)

url = "https://www.kurly.com/goods/1001432498"
test = driver.get(url)
time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")
reviews = soup.select("p.css-y49dcn.e36z05c13")
review_lists = [review.get_text().strip() for review in reviews]
# p 태그들을 없애고, 텍스트만 얻게 하기 위해서

driver.quit()

#엑셀 저장
wb = Workbook()
ws = wb.active
ws.title = "Kurly Reviews"
ws.append(["리뷰내용"])

for item in review_lists :
    ws.append([item]) # 리스트 형태로 넣어야 한 줄씩 추가 가능

okt = Okt() #형태소 구분할 객체 생성
all_text = " ".join(review_lists) # 여백을 기준으로 모든 값들을 하나의 문자로 가져오는 것
nouns = okt.nouns(all_text) # 명사만 찾아와주는 역할을 해주는 것
filtered_nouns = [n for n in nouns if len(n) > 1] #리스트 컴프리헨션, 조건문을 한 줄에 써주게 해주는 임시 변수
count = Counter(filtered_nouns) #인자값 안에 문자들이 몇 번 나오는지 카운트 해주는 객체

ws_keywords = wb.create_sheet("keyword Frequency") # 새로운 시트 생성
ws_keywords.append(["키워드", "빈도수"])
# 찾아온 카운트의 1-?의까지만 찾아오라는 명령어
for word, freq in count.most_common(30) : 
    ws_keywords.append([word, freq])
    
wb.save("kurly_reviews.xlsx")
print("엑셀 저장 완료 : kurly_reviews.xlsx" )

엑셀 저장 완료 : kurly_reviews.xlsx


In [30]:
from openpyxl import load_workbook
from konlpy.tag import Okt
from collections import Counter
import re

wb = load_workbook("연관키워드 20250722 1449.xlsx")
ws = wb.active

texts = [row[0] for row in ws.iter_rows(min_row = 2, max_col = 1, values_only = True) if row[0]]
# iter_rows > 범위 안에 있는 값을 찾아내서 리스트 형태로 반환, 행 단위로 반복할 수 있게 해주는 함수
# if row[0] > 값이 존재할 때만 필터링 (빈 셀은 제외한다)

okt = Okt()
all_text = " ".join(texts) #join: 한 칸씩 뗀 문자열로 바꾸어서 text에 넣어놓자
nouns_tokens = okt.nouns(all_text)

#정규표현식 #어떠한 패턴에 매칭되어 있는 값을 찾아라
regex_tokens = re.findall(r'[A-Za-z]*\d+[A-Za-z]+|\d+[A-Za-z]*', all_text) 
# A-Za-z 문자열 중에 알파벳이 있을 수도 없을 수도 있다 *은 0개 이상 반복
# \d 숫자 하나 (0~9)를 의미
# 마지막 + : 플래그 / 1개 이상, 0개 이상


#예외사항 만들기, 튜플로 만듦
valid_one_letter = {"금", "은", "동"}
stopwords = {"사용", "정도", "조금", "제품", "때문", "생각", "만족", "구매", "다음"}


combined_tokens = nouns_tokens + regex_tokens
filtered_tokens = [t for t in combined_tokens if (len(t) > 1 or t in valid_one_letter) and t not in stopwords]
#

count = Counter(filtered_tokens)


ws_keywords = wb.create_sheet("키워드 전체분석")
ws_keywords.append(["명사", "빈도수"])

for word, freq in sorted(count.items(), key = lambda x: x[1], reverse = True) :
    ws_keywords.append([word, freq])
#sorted() : 내림차순 정렬
#lambda : 익명함수, 실행문을 즉각적으로 쓸 수 있는 것     

wb.save("연관키워드_분석결과.xlsx")
print("완료: 전체 키워드 분석 결과 저장됨")

완료: 전체 키워드 분석 결과 저장됨


In [27]:
from openpyxl import Workbook
from konlpy.tag import Okt
from collections import Counter

wb = load_workbook("연관키워드 20250722 1551.xlsx")
ws = wb.active

texts = [row[0] for row in ws.iter_rows(min_row = 2, max_col = 1, values_only = True) if row[0]]

all_text = " ".join(texts)
nouns = okt.nouns(all_text)
filtered_nouns = [n for n in nouns if len(n) > 1]
count = Counter(filtered_nouns)

ws_keywords = wb.create_sheet("키워드 분석")
ws_keywords.append(["명사", "빈도수"])

for word, freq in sorted (count.items(), key = lambda x:x[1], reverse = True) :
    ws_keywords.append([word, freq])

wb.save("OTT 연관키워드 검색결과.xlsx")
print("저장 완료")

저장 완료
